In [1]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [3]:
df_ejercicios = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding = "utf-8")

In [16]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

## Incorporar clasificación de optimización

In [4]:
df_opt = pd.read_csv(root / "csv/ejercicios_optimizacion.csv", encoding = "utf-8")

In [5]:
df_opt.head()

,anio,convocatoria,ejercicio,enunciado_resumen,tema_optimizacion,categoria,max_min,cilindro
0,2007,Junio,Ejercicio 1 - Opción A,Dos números reales positivos cuya suma es 10 y...,Producto de los cuadrados de dos números con s...,Números,Máximo,No
1,2007,Reserva 1,Ejercicio 1 - Opción B,Dos chapas cuadradas de materiales a 2 y 3 €/c...,Coste total de dos chapas cuadradas con suma d...,Coste de materiales (figuras planas),Mínimo,No
2,2007,Reserva 2,Ejercicio 1 - Opción A,Rectángulos en el primer cuadrante con dos lad...,Área de un rectángulo inscrito entre una recta...,Rectángulo inscrito entre gráficas/funciones,Máximo,No
3,2007,Reserva 4,Ejercicio 1 - Opción B,Depósito en forma de prisma de base cuadrada s...,Superficie de un depósito prismático de base c...,Prisma/caja,Mínimo,No
4,2008,Junio,Ejercicio 1 - Opción B,Rectángulos de perímetro 8 cm. Dimensiones del...,Diagonal de un rectángulo con perímetro fijo,Rectángulo (diagonal),Mínimo,No


In [6]:
df_opt = df_opt.iloc[:, :-2]

In [7]:
df_opt.loc[:, ["asignatura", "tema", "tipo_ejercicio"]] = ["Mates II", "funciones", "optimización"]

In [8]:
df_opt = df_opt.rename(columns = {
    "anio" : "año",
    "categoria" : "observaciones"
})

In [9]:
df_opt.columns

Index(['año', 'convocatoria', 'ejercicio', 'enunciado_resumen',
       'tema_optimizacion', 'observaciones', 'asignatura', 'tema',
       'tipo_ejercicio'],
      dtype='str')

In [10]:
df_opt.drop(["enunciado_resumen", "tema_optimizacion"], axis = 1, inplace=True)

In [17]:
def create_id(row):
    row.ejercicio = row.ejercicio.replace(" - Opción", " opción")
    id = f"{row.asignatura}_{row.tema}_{row.año}_{row.convocatoria}_{row.ejercicio}".lower().replace(" ", "_")
    id = remove_accents(id)
    return id

In [18]:
df_opt["id"] = df_opt.apply(create_id, axis = 1)

In [19]:
df_opt.head(2)

,año,convocatoria,ejercicio,observaciones,asignatura,tema,tipo_ejercicio,id
0,2007,Junio,Ejercicio 1 - Opción A,Números,Mates II,funciones,optimización,mates_ii_funciones_2007_junio_ejercicio_1_opci...
1,2007,Reserva 1,Ejercicio 1 - Opción B,Coste de materiales (figuras planas),Mates II,funciones,optimización,mates_ii_funciones_2007_reserva_1_ejercicio_1_...


In [20]:
df_opt_to_merge = df_opt.loc[:, ["id", "tipo_ejercicio", "observaciones"]]

In [21]:
merged_df = pd.merge(df_ejercicios, df_opt_to_merge, on = "id", how = "left")

Combinar los valores de `tipo_ejercicio`

In [22]:
def get_exercise_type(row):
    tipos = [row.tipo_ejercicio_x, row.tipo_ejercicio_y]
    tipo  = [i for i in tipos if pd.notna(i)]
    if len(tipo) > 0:
        return tipo[0]
    else:
        return pd.NA


In [23]:
merged_df["tipo_ejercicio"] = merged_df.apply(get_exercise_type, axis = 1)
merged_df.drop(["tipo_ejercicio_x", "tipo_ejercicio_y"], axis = 1, inplace=True)

## Merge de ejercicios clasificados y optimización

In [ ]:
# merged_df.to_csv("csv/ejercicios_clasificados.csv", index = False, encoding="utf-8")

In [ ]:
merged_df.sample(3)

,asignatura,año,convocatoria,ejercicio,tema,enunciado,id,info_ejercicio,archivo,imagen,observaciones,tipo_ejercicio
1783,Mates II,2019,Junio,"Ejercicio 1, opción B",funciones,Considera la función f :  definida por f(x)(...,mates_ii_funciones_2019_junio_ejercicio_1_opci...,MATEMÁTICAS II. 2019. JUNIO. EJERCICIO 1. OPCI...,source_pdf_files/Mates II/funciones/2019 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...,NaN,NaN
2907,Mates II,2009,Reserva 4,"Ejercicio 3,opción B",sistemas de ecuaciones lineales,x y  m1\n\nSea el sistema de ecuaciones l...,mates_ii_sistemas_de_ecuaciones_lineales_2009_...,MATEMÁTICAS II. 2009. RESERVA 4. EJERCICIO 3.O...,source_pdf_files/Mates II/sistemas_de_ecuacion...,enunciados_ejercicios/Mates II/sistemas_de_ecu...,NaN,NaN
2811,Mates II,2024,Modelo,Ejercicio 5,sistemas de ecuaciones lineales,Dadas las matrices A =\n 1\n1\n0\n1\n0\n1...,mates_ii_sistemas_de_ecuaciones_lineales_2024_...,MATEMÁTICAS II. 2024. MODELO. EJERCICIO 5,source_pdf_files/Mates II/sistemas_de_ecuacion...,enunciados_ejercicios/Mates II/sistemas_de_ecu...,NaN,NaN
